# Part 7 — Advanced Topics

*Last updated:* 2026-01-08

This notebook collects advanced, production-oriented topics for power users:

- custom external database profiles (strict vs broad)
- programmatic YAML management (useful in pipelines)
- troubleshooting & diagnostics (common failure modes)
- integration patterns (Snakemake/Nextflow-friendly workflows)

> **Tip:** You don’t need this notebook for day-to-day conversions. It’s here for when you want reproducibility at scale.


## 7.1 — Custom External Database Inclusion

The external YAML is an explicit **contract**: it defines which external namespaces are allowed to influence your graph.

Two common profiles:

- **Strict profile (recommended for most analyses):** small allowlist, lower ambiguity, faster queries.
- **Broad profile (exploration):** larger allowlist, more coverage, but higher ambiguity and slower builds.

In IDTrack today, the active YAML file name is fixed (`<organism>_externals_modified.yml`).
A practical pattern is to keep multiple profiles as *side-by-side files* and copy/rename the one you want for a given run.

The next cell demonstrates how to generate two profile files **without overwriting** your active YAML.


In [ ]:
from __future__ import annotations

import os
from copy import deepcopy
from pathlib import Path

import yaml

try:
    import idtrack

    IDTRACK_OK = True
except Exception as e:
    print('idtrack import failed ->', repr(e))
    IDTRACK_OK = False

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

print('Local repository:', LOCAL_REPOSITORY)

if IDTRACK_OK:
    organism = 'homo_sapiens'

    # Prefer your configured YAML if it exists; otherwise fall back to the package default.
    configured = LOCAL_REPOSITORY / f'{organism}_externals_modified.yml'
    default_cfg = Path(idtrack.__file__).resolve().parent / 'default_config' / f'{organism}_externals_modified.yml'

    source_path = configured if configured.exists() else default_cfg
    print('Reading YAML from:', source_path)

    y = yaml.safe_load(source_path.read_text(encoding='utf-8'))
    form = list(y[organism].keys())[0]

    strict_allowlist = {
        'HGNC Symbol',
        'EntrezGene',
        'UniProtKB',
        'RefSeq_mRNA',
    }

    broad_allowlist = strict_allowlist | {
        # Add cautiously; broad profiles can increase ambiguity.
        'RefSeq_peptide',
        'ArrayExpress',
    }

    def make_profile(base: dict, allowlist: set[str]) -> dict:
        out = deepcopy(base)
        for db_name, db_block in out[organism][form].items():
            include = db_name in allowlist
            for _asm, attrs in db_block.get('Assembly', {}).items():
                attrs['Include'] = bool(include)
        return out

    strict_yaml = make_profile(y, strict_allowlist)
    broad_yaml = make_profile(y, broad_allowlist)

    strict_path = LOCAL_REPOSITORY / f'{organism}_externals_modified_strict.yml'
    broad_path = LOCAL_REPOSITORY / f'{organism}_externals_modified_broad.yml'

    strict_path.write_text(yaml.safe_dump(strict_yaml, sort_keys=False, allow_unicode=True), encoding='utf-8')
    broad_path.write_text(yaml.safe_dump(broad_yaml, sort_keys=False, allow_unicode=True), encoding='utf-8')

    print('Wrote strict profile:', strict_path.name)
    print('Wrote broad  profile:', broad_path.name)

    print()
    print('To activate a profile: copy/rename it to:')
    print(' ', configured)
else:
    print('Skipping YAML profile demo (idtrack not imported).')


## 7.2 — Programmatic YAML Management

For pipelines you often want YAML changes to be **scriptable** and **repeatable**.

A safe automation pattern:

1. Generate (or refresh) a template YAML.
2. Apply a curated allowlist in code.
3. Write the resulting YAML to a file you commit alongside your pipeline.

The next cell shows a reusable helper that:
- reads a template YAML
- applies an allowlist
- writes a modified YAML

> **Tip:** Keep allowlists per organism in your pipeline repository (as Python sets or a small TOML/YAML). That makes your configuration reviewable.


In [ ]:
from __future__ import annotations

import os
from copy import deepcopy
from pathlib import Path

import yaml


def apply_allowlist_to_yaml(template_path: Path, organism: str, allowlist: set[str], out_path: Path) -> None:
    y = yaml.safe_load(template_path.read_text(encoding='utf-8'))
    form = list(y[organism].keys())[0]

    out = deepcopy(y)
    enabled = []

    for db_name, db_block in out[organism][form].items():
        include = db_name in allowlist
        if include:
            enabled.append(db_name)
        for _asm, attrs in db_block.get('Assembly', {}).items():
            attrs['Include'] = bool(include)

    out_path.write_text(yaml.safe_dump(out, sort_keys=False, allow_unicode=True), encoding='utf-8')
    print('Wrote:', out_path)
    print('Enabled (count):', len(enabled))
    print('Enabled (sample):', enabled[:15])


# Example: apply a mouse allowlist if a template exists in your local repository
LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
organism = 'mus_musculus'

template = LOCAL_REPOSITORY / f'{organism}_externals_template.yml'
out_yaml = LOCAL_REPOSITORY / f'{organism}_externals_modified.yml'

allowlist_mouse = {'MGI Symbol', 'EntrezGene', 'UniProtKB', 'RefSeq_mRNA'}

if template.exists():
    apply_allowlist_to_yaml(template, organism=organism, allowlist=allowlist_mouse, out_path=out_yaml)
else:
    print('Template not found:', template)
    print('Generate it first with Part 2 (prepare_new_external_yaml.ipynb).')


## 7.3 — Troubleshooting & Diagnostics

Common failure modes (and what they usually mean):

- **Permission errors in `IDTRACK_LOCAL_REPO`:** your cache directory is not writable.
- **REST timeouts:** network issues or Ensembl REST is temporarily slow.
- **MySQL connection errors:** firewall/VPN issues (ports `3306/5306/3337`).
- **`ValueError: release not included in YAML`:** your external YAML does not include the snapshot boundary you chose.
- **Unexpected 1→n explosion:** your external allowlist is too broad or contains promiscuous namespaces.

The next cell is a compact diagnostic report you can paste into issues or lab notes.


In [ ]:
from __future__ import annotations

import os
import socket
from pathlib import Path

report = {}

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

report['local_repository'] = str(LOCAL_REPOSITORY)
report['local_repository_writable'] = os.access(LOCAL_REPOSITORY, os.W_OK)

# YAML + graph snapshot inventory
report['yaml_files'] = sorted(p.name for p in LOCAL_REPOSITORY.glob('*_externals_modified.yml'))
report['graph_snapshots'] = sorted(p.name for p in LOCAL_REPOSITORY.glob('graph_*.pickle'))

# REST connectivity
try:
    import requests

    try:
        r = requests.get('https://rest.ensembl.org/info/ping', headers={'Content-Type': 'application/json'}, timeout=15)
        report['ensembl_rest_status'] = r.status_code
    except Exception as e:
        report['ensembl_rest_status'] = f'failed: {e.__class__.__name__}'
except Exception as e:
    report['ensembl_rest_status'] = f'requests_missing: {e.__class__.__name__}'

# MySQL ports (best-effort)
try:
    from idtrack._db import DB

    host = DB.mysql_host
    port_status = {}
    for port in [3306, 5306, 3337]:
        try:
            with socket.create_connection((host, port), timeout=2):
                port_status[port] = 'ok'
        except OSError as e:
            port_status[port] = e.__class__.__name__
    report['ensembl_mysql_ports'] = port_status
except Exception as e:
    report['ensembl_mysql_ports'] = f'skipped: {e.__class__.__name__}'

# Print report
for k, v in report.items():
    print(f'{k}: {v}')

# Optional: quick integrity checks (only if a human snapshot exists)
# NOTE: TrackTests can be expensive. We only run a very small, cheap check here.
if report['graph_snapshots'] and any('homo_sapiens' in g for g in report['graph_snapshots']):
    try:
        import idtrack

        api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
        api.configure_logger()
        org, latest = api.resolve_organism('human')

        # Load existing snapshot if present; build if missing (may be slow).
        api.build_graph(organism_name=org, snapshot_release=latest, return_test=True, calculate_caches=False)

        ok = api.track.is_edge_with_same_nts_only_at_backbone_nodes()
        print()
        print('Quick TrackTests check (edge nts invariant):', ok)
    except Exception as e:
        print()
        print('TrackTests quick check skipped/failed ->', repr(e))


## 7.4 — Integration Patterns

IDTrack works best in pipelines when you make the snapshot boundary and cache location explicit.

Key ideas:

- Set `IDTRACK_LOCAL_REPO` to a stable, shared path (per project or per compute environment).
- Build snapshots once, then reuse them across jobs.
- Store your external YAML alongside your pipeline so the configuration is reviewable.

Below are lightweight patterns you can adapt.


In [ ]:
# Snakemake / Nextflow snippets (printed as plain text)

snakemake_rule = """
rule idtrack_build_human_graph:
    output:
        'idtrack_cache/graph_homo_sapiens_*.pickle'
    shell:
        "python - <<'PY'\nimport os\nfrom pathlib import Path\nimport idtrack\n\nos.environ['IDTRACK_LOCAL_REPO'] = os.path.abspath('idtrack_cache')\nPath(os.environ['IDTRACK_LOCAL_REPO']).mkdir(parents=True, exist_ok=True)\n\napi = idtrack.API(local_repository=os.environ['IDTRACK_LOCAL_REPO'])\norg, latest = api.resolve_organism('human')\napi.build_graph(organism_name=org, snapshot_release=latest, calculate_caches=True)\nprint('Built:', org, latest)\nPY"
""".strip()

nextflow_process = """
process BUILD_IDTRACK_HUMAN {
  output:
    path "idtrack_cache/graph_homo_sapiens_*.pickle"
  script:
    "export IDTRACK_LOCAL_REPO=$PWD/idtrack_cache\npython - <<'PY'\nimport os\nfrom pathlib import Path\nimport idtrack\n\nrepo = os.environ['IDTRACK_LOCAL_REPO']\nPath(repo).mkdir(parents=True, exist_ok=True)\n\napi = idtrack.API(local_repository=repo)\norg, latest = api.resolve_organism('human')\napi.build_graph(organism_name=org, snapshot_release=latest, calculate_caches=True)\nprint('Built:', org, latest)\nPY"
}
""".strip()

print('--- Snakemake example ---')
print(snakemake_rule)
print('--- Nextflow example ---')
print(nextflow_process)
